# 01 First look

A first check that the setup works on real data, and a first try at the views from the plan:

1. what is in place (data, weights, device)
2. the labels: how many per site and class, and how long they are (annotators differ between site-years)
3. all three models on a few dev recordings, scored with the official scorer
4. which model finds which labelled call (a first, simple version of V4)
5. one timeline per class with every model's output (a first version of V5), with a slider if ipywidgets is installed

Running the models takes a few minutes on a laptop CPU (two one-hour recordings per dev site). The results go to `outputs/<model>/validation_firstlook`, so the next run of the notebook just loads them.

In [ ]:
import json, time
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import stft
import torch

from aivis import paths, io, evaluate, predict
from aivis.recordings import file_start

C = {"yolo": "#1F6FB2", "whalevad": "#D9822B", "nave": "#7B3F9E", "label": "#9A9A9A"}
NAMES = {"yolo": "YOLO baseline", "whalevad": "Whale-VAD", "nave": "NAVE"}
CLASSES = paths.CLASSES
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

## 1. What is in place

In [ ]:
rows = []
for split, base in (("train", paths.DEV / "train"), ("validation", paths.DEV / "validation"), ("evaluation", paths.EVAL)):
    for site in paths.SITES[split]:
        rows.append({"split": split, "site": site, "wav files": len(list((base / "audio" / site).glob("*.wav"))),
                     "labels file": (base / "annotations" / f"{site}.csv").exists()})
print(pd.DataFrame(rows).to_string(index=False))
for key in paths.CONFIG["weights"]:
    print(f"{key:14} {'ok' if paths.weights(key).exists() else 'MISSING'}  {paths.weights(key).name}")
mps = getattr(torch.backends, "mps", None)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| mps", bool(mps and mps.is_available()))

## 2. The labels

All labels of the training and dev sites, joined to the three classes. The duration plot is the cheap check from the plan: if one site's annotator draws much longer or shorter boxes for the same call type, IoU-based scores carry that annotator's style.

In [ ]:
parts = []
for split in ("train", "validation"):
    for site in paths.SITES[split]:
        df = pd.read_csv(paths.annotations_dir(split) / f"{site}.csv")
        df["split"] = split
        parts.append(df)
labels = pd.concat(parts, ignore_index=True)
labels["class"] = labels["annotation"].map(paths.FINE_TO_CLASS)
labels["start"] = pd.to_datetime(labels["start_datetime"], format="ISO8601", utc=True)
labels["end"] = pd.to_datetime(labels["end_datetime"], format="ISO8601", utc=True)
labels["duration_s"] = (labels["end"] - labels["start"]).dt.total_seconds()
counts = labels.pivot_table(index=["split", "dataset"], columns="class", values="duration_s", aggfunc="size", fill_value=0)[CLASSES]
counts

In [ ]:
sites = paths.SITES["train"] + paths.SITES["validation"]
fig, axes = plt.subplots(1, 3, figsize=(11, 4.2), sharey=True)
for ax, cls in zip(axes, CLASSES):
    data = [labels.loc[(labels.dataset == s) & (labels["class"] == cls), "duration_s"].values for s in sites]
    keep = [i for i, d in enumerate(data) if len(d)]
    horiz = {"orientation": "horizontal"} if tuple(map(int, matplotlib.__version__.split(".")[:2])) >= (3, 10) else {"vert": False}
    bp = ax.boxplot([data[i] for i in keep], positions=keep, widths=0.6, showfliers=False,
                    patch_artist=True, medianprops={"color": "black", "lw": 1.5}, **horiz)
    for i, box in zip(keep, bp["boxes"]):
        box.set(facecolor="#D9D9D9" if sites[i] in paths.SITES["train"] else "#8C8C8C", edgecolor="none")
    ax.set_title(cls)
    ax.set_xlabel("label duration (s)")
axes[0].set_yticks(range(len(sites)), sites)
axes[0].invert_yaxis()
fig.suptitle("Label durations per site (light: training sites, dark: dev sites), outliers hidden", x=0.01, ha="left")
fig.tight_layout()

In [ ]:
med = labels.pivot_table(index="dataset", columns="class", values="duration_s", aggfunc="median")[CLASSES].round(1)
med.loc[sites]

## 3. The three models on a few dev recordings

The recordings per dev site with the most labels, so there is something to look at. With a GPU this takes seconds, on a laptop CPU about a minute per recording and model. Change `N_PER_SITE` for more; the run folder then needs a new `TAG` (or delete the old one).

In [ ]:
N_PER_SITE = 2
TAG = "firstlook"
val = labels[labels.split == "validation"]
pick = (val.groupby(["dataset", "filename"]).size().rename("n").reset_index()
        .sort_values("n", ascending=False).groupby("dataset").head(N_PER_SITE).sort_values(["dataset", "filename"]))
pick

In [ ]:
runs = {}
for model in ("yolo", "whalevad", "nave"):
    out = paths.OUTPUTS / model / f"validation_{TAG}"
    if not (out / "detections.csv").exists():
        t0 = time.time()
        predict.main([model, "--split", "validation", "--sites", *pick.dataset.unique(),
                      "--files", *pick.filename, "--tag", TAG, "--overwrite"])
        print(f"{model}: {time.time() - t0:.0f} s")
    runs[model] = out
dets = {m: pd.read_csv(out / "detections.csv") for m, out in runs.items()}
{m: len(d) for m, d in dets.items()}

In [ ]:
gt = evaluate.load_labels("validation", list(pick.dataset.unique()),
                          [f"{d}/{f}" for d, f in zip(pick.dataset, pick.filename)])
scores = []
for m, d in dets.items():
    table, summary = evaluate.evaluate(d, gt)
    s = summary.set_index("dataset")
    scores.append({"model": NAMES[m], **{site: round(s.loc[site, "macro_B"], 3) for site in s.index}})
print("macro B (leaderboard F1) on these few recordings only, not comparable to the published numbers")
pd.DataFrame(scores).set_index("model")

## 4. Which model finds which labelled call

For every label in these recordings: does a model have a detection of the same class with IoU above 0.3? This is a simple matching for looking, not the official one (it allows one detection to cover two labels). The tall bars on the right are the calls that only one or two models find.

In [ ]:
def to_seconds(df):
    t0 = pd.to_datetime(df.filename.map(file_start), utc=True)
    s = (pd.to_datetime(df.start_datetime, format="ISO8601", utc=True) - t0).dt.total_seconds()
    e = (pd.to_datetime(df.end_datetime, format="ISO8601", utc=True) - t0).dt.total_seconds()
    return df.assign(s=s.values, e=e.values)

g = to_seconds(gt.assign(cls=gt.annotation.map(paths.FINE_TO_CLASS)))
dd = {m: to_seconds(d.rename(columns={"annotation": "cls"})) for m, d in dets.items()}

def hit(row, d):
    c = d[(d.dataset == row.dataset) & (d.filename == row.filename) & (d.cls == row.cls)]
    inter = (np.minimum(c.e, row.e) - np.maximum(c.s, row.s)).clip(lower=0)
    union = (np.maximum(c.e, row.e) - np.minimum(c.s, row.s))
    return bool(((inter / union) > 0.3).any())

for m in dd:
    g[m] = [hit(r, dd[m]) for r in g.itertuples()]
g["found by"] = g[list(dd)].apply(lambda r: " + ".join(NAMES[m] for m in dd if r[m]) or "none", axis=1)
combo = g.groupby(["cls", "found by"]).size().unstack(0, fill_value=0).reindex(columns=CLASSES, fill_value=0)
combo = combo.loc[combo.sum(axis=1).sort_values().index]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
for ax, cls in zip(axes, CLASSES):
    ax.barh(range(len(combo)), combo[cls], color="#5A5A5A", height=0.7)
    for i, v in enumerate(combo[cls]):
        if v:
            ax.text(v, i, f" {v}", va="center", fontsize=8)
    ax.set_title(f"{cls} ({combo[cls].sum()} labels)")
    ax.set_xlabel("labelled calls")
axes[0].set_yticks(range(len(combo)), combo.index)
fig.suptitle("Which models find each labelled call (IoU > 0.3, same class)", x=0.01, ha="left")
fig.tight_layout()

## 5. Every output on one timeline

Top: the spectrogram (log magnitude) with the label boxes. Below it one row per class: grey spans are labels, the blue bars are YOLO detections, the orange and purple lines are the Whale-VAD and NAVE frame probabilities with their thresholds dashed. Pick an event from the list to jump to it.

In [ ]:
probs_cache = {}
def frame_probs(model, site, fn):
    key = (model, site, fn)
    if key not in probs_cache:
        f = runs[model] / "frame_probs" / site / fn.replace(".wav", ".npz")
        probs_cache[key] = io.load_frame_probs(f)["probs"].astype(float) if f.exists() else None
    return probs_cache[key]

thr = {m: json.loads((runs[m] / "run.json").read_text())["settings"]["thresholds"] for m in ("whalevad", "nave")}

def show(site, fn, t_center, span=90):
    t0, t1 = max(0, t_center - span / 2), t_center + span / 2
    path = paths.audio_dir("validation") / site / fn
    with sf.SoundFile(str(path)) as f:
        sr = f.samplerate
        f.seek(int(t0 * sr))
        x = f.read(int((t1 - t0) * sr), dtype="float32", always_2d=True)[:, 0]
    fr, tt, Z = stft(x, fs=sr, nperseg=256, noverlap=246)
    S = 20 * np.log10(np.abs(Z) + 1e-9)
    fig, axes = plt.subplots(4, 1, figsize=(11, 7.5), sharex=True, gridspec_kw={"height_ratios": [2.4, 1, 1, 1]})
    ax = axes[0]
    ax.pcolormesh(tt + t0, fr, S, cmap="Greys", shading="auto", vmin=np.percentile(S, 30), vmax=np.percentile(S, 99.5))
    lab = g[(g.dataset == site) & (g.filename == fn) & (g.e > t0) & (g.s < t1)]
    for r in lab.itertuples():
        ax.add_patch(plt.Rectangle((r.s, r.low_frequency), r.e - r.s, r.high_frequency - r.low_frequency,
                                   fill=False, ec="#2E8B57", lw=1.4))
        ax.text(r.s, r.high_frequency + 1, r.annotation, color="#2E8B57", fontsize=7)
    yb = dd["yolo"]
    yb = yb[(yb.dataset == site) & (yb.filename == fn) & (yb.e > t0) & (yb.s < t1)]
    for r in yb.itertuples():
        ax.add_patch(plt.Rectangle((r.s, r.low_frequency), r.e - r.s, r.high_frequency - r.low_frequency,
                                   fill=False, ec=C["yolo"], lw=1.2, ls="--"))
    ax.set_ylabel("Hz")
    ax.set_title(f"{site} / {fn}, {t0:.0f} to {t1:.0f} s   (green: labels, blue dashed: YOLO boxes)", loc="left")
    for k, cls in enumerate(CLASSES):
        ax = axes[k + 1]
        for r in lab[lab.cls == cls].itertuples():
            ax.axvspan(r.s, r.e, color=C["label"], alpha=0.35, lw=0)
        for r in yb[yb.cls == cls].itertuples():
            ax.plot([r.s, r.e], [1.04, 1.04], color=C["yolo"], lw=4, solid_capstyle="butt")
        for m in ("whalevad", "nave"):
            p = frame_probs(m, site, fn)
            if p is None:
                continue
            f0, f1 = int(t0 / 0.02), min(int(t1 / 0.02), len(p))
            ax.plot(np.arange(f0, f1) * 0.02, p[f0:f1, k], color=C[m], lw=1.2, label=NAMES[m])
            ax.axhline(thr[m][cls], color=C[m], lw=0.8, ls="--")
        ax.set_ylim(0, 1.1)
        ax.set_ylabel(cls)
    axes[1].plot([], [], color=C["yolo"], lw=4, label=NAMES["yolo"])
    axes[1].fill_between([], [], color=C["label"], alpha=0.35, label="label")
    axes[1].legend(loc="upper right", fontsize=7, ncol=4, frameon=False)
    axes[-1].set_xlabel("seconds from the start of the file")
    axes[0].set_xlim(t0, t1)
    fig.tight_layout()
    plt.show()

In [ ]:
events = g.sort_values(["dataset", "filename", "s"]).reset_index(drop=True)
events["what"] = [f"{r.dataset} {r.filename[:16]} {r.s:7.1f}s {r.cls:5} found by: {fb}"
                  for r, fb in zip(events.itertuples(), events["found by"])]
try:
    import ipywidgets as w
    pickw = w.Dropdown(options=list(zip(events.what, events.index)), description="event", layout=w.Layout(width="95%"))
    spanw = w.IntSlider(value=90, min=20, max=300, step=10, description="window (s)")
    w.interact(lambda i, span: show(events.dataset[i], events.filename[i], (events.s[i] + events.e[i]) / 2, span),
               i=pickw, span=spanw)
except ImportError:
    print("pip install ipywidgets for the dropdown; showing three examples instead")
    for i in [0, len(events) // 2, len(events) - 1]:
        show(events.dataset[i], events.filename[i], (events.s[i] + events.e[i]) / 2)

In [ ]:
# a call that only some models find, if there is one
miss = events[events["found by"].str.count(r"\+") < 2]
if len(miss):
    r = miss.iloc[0]
    print(r.what)
    show(r.dataset, r.filename, (r.s + r.e) / 2)